In [24]:
import pandas as pd
import numpy as np

In [52]:
full_old = pd.read_csv("../files/stats/lits/old_summary.csv", dtype=str)
full_new = pd.read_csv("../files/stats/lits/per_case_summary.csv", dtype=str)

In [3]:
columns_to_exclude = ['case_name', 'lesion_sphericities',
                      'liver_hu_p01', 'liver_hu_p99', 'tumour_volume_ml', 'voxel_volume_mm3',
                      'liver_volume_ml', 'lesion_volumes_ml', 'lesion_equiv_diameters_mm',
                      'min_lesion_diameter_mm', 'max_lesion_diameter_mm', 
                      'mean_lesion_diameter_mm', ]

In [4]:
columns_to_update = [col for col in full_old.columns if col not in columns_to_exclude]

update_map = full_new.set_index('case_name')[columns_to_update]

for col in columns_to_update:
    full_old[col] = full_old['case_name'].map(update_map[col])

In [5]:
full_old.to_csv('../files/stats/lits/old_summary_2.csv', index=False)

## Agregar elementos

Ahora que ya sabemos cuáles son las columnas que han cambiado, podemos
checar si son relevantes

In [ ]:
columns_to_check = [c for c in columns_to_exclude if c != 'lesion_sphericities']

# 2. Validate all required columns exist before sub-setting
missing = set(columns_to_check) - set(full_new.columns)
if missing:
    raise ValueError(f"Columns not found in full_new: {missing}")

full_to_check = full_new[columns_to_check]

full_to_check

,case_name,liver_hu_p01,liver_hu_p99,tumour_volume_ml,voxel_volume_mm3,liver_volume_ml,lesion_volumes_ml,lesion_equiv_diameters_mm,min_lesion_diameter_mm,max_lesion_diameter_mm,mean_lesion_diameter_mm
0,volume-0.nii.gz,28.0,182.0,6.404754638671875,2.471923828125,1359.6470947265625,0.11370849609375;0.106292724609375;0.242248535...,6.010788282186921;5.877170712552611;7.73429701...,5.877170712552611,17.238073703714516,9.256542345636966
1,volume-1.nii.gz,25.0,180.0,16.426778722835888,2.283399878070042,1470.004890104054,0.3858945793938372;0.3379431819543663;0.182671...,9.032819915873212;8.642015127762834;7.03975880...,7.039758800600867,20.23187370516639,12.717312940716944
2,volume-10.nii.gz,6.0,164.0,14.395855560302737,0.5921783447265625,1692.070268157959,0.32806680297851565;8.628038482666016;3.713550...,8.556993879850493;25.447071909229226;19.213075...,7.141179074100664,25.447071909229223,13.844598619977509
3,volume-100.nii.gz,8.0,209.0,537.2021496118745,0.34223476727675,1712.2248393540572,2.5013939140257664;532.1045627532874;0.1574279...,16.842032025878854;100.53858923681194;6.699274...,3.471377356049093,100.53858923681194,16.746920935597444
4,volume-101.nii.gz,13.0,203.0,131.69473135138352,0.347994607721148,2255.5587174539232,30.34652177171499;0.6907692963264788;0.0003479...,38.69932880971558;10.96759658733492;0.87268590...,0.8726859099170995,50.28923372352221,13.41995625364898
...,...,...,...,...,...,...,...,...,...,...,...
126,volume-95.nii.gz,-2.0,200.0,1.1245808738428174,0.4357151777771473,1772.5403218732354,1.1219665827761545;0.0008714303555542947;0.001...,12.892201948471337;1.1850718641721716;1.493096...,1.1850718641721716,12.892201948471335,5.190123599984061
127,volume-96.nii.gz,16.0,206.0,15.794052787364327,0.3896591120164885,2225.169790421319,0.1628775088228922;0.4882428673566601;0.036627...,6.775699942821541;9.769652019236547;4.12038713...,4.120387135763136,21.90213562445832,9.254127334810908
128,volume-97.nii.gz,17.0,215.0,265.1460880901798,0.3655624327393525,1677.2454055873764,79.75585263804275;26.967175100749298;65.604930...,53.40574527231217;37.20593967623354;50.0394606...,6.374492512998745,53.40574527231217,26.733058262172538
129,volume-98.nii.gz,23.0,237.0,229.7701341973894,0.3775146420084638,1326.9148897562893,7.626173283212978;0.35259867563590525;0.164218...,24.421332067375147;8.765176885971075;6.7942493...,6.301175141203033,69.27158140238046,16.831636450278125


In [16]:
def mix_old_and_new(column_name: str, newest_df: pd.DataFrame, old_df: pd.DataFrame) -> pd.DataFrame:
    # Extract and rename
    aux_df = newest_df[['case_name', column_name]].rename(
        columns={column_name: "new"}
    )

    # SAFE index-aligned merge (no positional assumption)
    old_lookup = old_df.set_index('case_name')[column_name]
    aux_df['old'] = aux_df['case_name'].map(old_lookup)

    # 6. Validate no unmatched cases introduced NaN
    unmatched = aux_df[aux_df['old'].isna()]
    if not unmatched.empty:
        print("Warning: %d Unmatched cases found:" % len(unmatched))
        print(unmatched['case_name'].tolist())

    return aux_df

In [33]:
def compare_change(column_name: str, newest_df: pd.DataFrame, oldest_df: pd.DataFrame) -> pd.DataFrame:
    # 1. Load and coerce to numeric
    aux_df = mix_old_and_new(column_name, newest_df, oldest_df)
    aux_df['new'] = pd.to_numeric(aux_df['new'], errors='coerce')
    aux_df['old'] = pd.to_numeric(aux_df['old'], errors='coerce')

    # 2. Vectorised difference
    aux_df['difference'] = aux_df['new'] - aux_df['old']

    # 3. Vectorised percent change
    # np.where evaluates the condition array at C-speed, avoiding the Python loop
    aux_df['percent_change'] = np.where(
        aux_df['old'] != 0,
        (aux_df['difference'] / aux_df['old']) * 100,
        np.nan
    )

    # 4. Sort and ASSIGN back to the variable
    aux_df = aux_df.sort_values(
        by='percent_change', 
        key=abs,             # The built-in abs() function works natively on pandas Series
        ascending=False,
        na_position='last'   # Pushes cases with no old tumour (NaN) to the bottom
    )

    # Filter out NaN values for statistical summary
    valid_changes = aux_df['percent_change'].dropna()

    print(f"Mean absolute percent change: {valid_changes.abs().mean():.4f}%")
    print(f"Max absolute percent change:  {valid_changes.abs().max():.4f}%")

    # If the max change is microscopic, it scientifically justifies keeping the old split
    if valid_changes.abs().max() < 1.0:
        print("Conclusion: Variations are sub-clinical. Retaining old stratification is justified.")
    else:
        print("Conclusion: Significant variations detected. Consider re-evaluating stratification.")

    return aux_df

In [19]:
columns_to_check

['case_name',
 'liver_hu_p01',
 'liver_hu_p99',
 'tumour_volume_ml',
 'voxel_volume_mm3',
 'liver_volume_ml',
 'lesion_volumes_ml',
 'lesion_equiv_diameters_mm',
 'min_lesion_diameter_mm',
 'max_lesion_diameter_mm',
 'mean_lesion_diameter_mm']

In [43]:
compare_change('mean_lesion_diameter_mm', full_new, full_old).head(3)

['volume-105.nii.gz', 'volume-106.nii.gz', 'volume-114.nii.gz', 'volume-115.nii.gz', 'volume-119.nii.gz', 'volume-32.nii.gz', 'volume-34.nii.gz', 'volume-38.nii.gz', 'volume-41.nii.gz', 'volume-47.nii.gz', 'volume-87.nii.gz', 'volume-89.nii.gz', 'volume-91.nii.gz']
Mean absolute percent change: 0.0000%
Max absolute percent change:  0.0000%
Conclusion: Variations are sub-clinical. Retaining old stratification is justified.


,case_name,new,old,difference,percent_change
118,volume-88.nii.gz,15.785802,15.785802,4.083362e-07,0.000003
47,volume-23.nii.gz,34.993816,34.993816,-6.000586e-07,-0.000002
15,volume-111.nii.gz,10.095589,10.095589,1.730859e-07,0.000002


### Columns to change:

After a thorough analysis, we can conclude that the only columns we need
to change / update are:

- liver_hu_p01
- liver_hu_p99

(That are the only columns that were originally modified, to begin with)

## Update the columns

In [49]:
def update_values(column_name: str, newest_df: pd.DataFrame, old_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Updates the values of a specified column in the old dataframe with those from the newest dataframe.

    Params
    ------
    `column_name`: str
        The name of the column to update. Must exist in both dataframes.
    `newest_df`: pd.DataFrame
        The dataframe containing the new values.
    `old_df`: pd.DataFrame
        The dataframe to be updated. Must contain a 'case_name' column for merging.
    '''
    aux_df = old_df.copy()
    new_lookup = newest_df.set_index('case_name')[column_name]
    
    # Map the new values into the copy of the old dataframe
    aux_df[column_name] = aux_df['case_name'].map(new_lookup)

    # Validate unmatched cases
    unmatched = aux_df[aux_df[column_name].isna()]
    if not unmatched.empty:
        print(f"Warning: {len(unmatched)} cases in old_df have no match in newest_df:")
        print(unmatched['case_name'].tolist())

    return aux_df

In [53]:
full_old

,case_name,image_shape,label_shape,ct_min,ct_max,spacing_x,spacing_y,spacing_z,affine_codes,unique_labels,...,tumour_volume_ml,num_lesions,lesion_volumes_ml,lesion_equiv_diameters_mm,lesion_sphericities,min_lesion_diameter_mm,max_lesion_diameter_mm,mean_lesion_diameter_mm,liver_texture_variance,liver_noise_estimate
0,volume-0.nii.gz,"(512, 512, 75)","(512, 512, 75)",-3024.0,1410.0,0.703125,0.703125,5.0,"('L', 'A', 'S')",0;1;2,...,6.404755115509033,11,0.11370849609375;0.106292724609375;0.242248535...,6.010788282186921;5.877170712552611;7.73429701...,0.20204138562726295;0.21613729625242084;0.0948...,5.877170712552611,17.238073703714516,9.256542345636966,468.3676758076976,15.555555555555555
1,volume-1.nii.gz,"(512, 512, 123)","(512, 512, 123)",-3024.0,3071.0,0.6757810115814209,0.6757810115814209,5.0,"('L', 'A', 'S')",0;1;2,...,16.426776885986328,12,0.38589456963539126;0.3379431734085083;0.18267...,9.032819839733;8.642015054916826;7.03975874126...,0.05646714695082087;0.07104050498133552;0.1192...,7.039758741260735,20.23187353462616,12.717312833519088,447.40374734311297,14.814814814814811
2,volume-10.nii.gz,"(512, 512, 501)","(512, 512, 501)",-1024.0,1602.0,0.76953125,0.76953125,1.0,"('L', 'A', 'S')",0;1;2,...,14.395855903625488,6,0.32806680297851565;8.628038482666016;3.713550...,8.556993879850493;25.447071909229226;19.213075...,0.04699701043515416;0.009242349629454267;0.015...,7.141179074100664,25.447071909229223,13.844598619977509,754.6303296819981,25.185185185185183
3,volume-100.nii.gz,"(512, 512, 685)","(512, 512, 685)",-1024.0,3071.0,0.69921875,0.69921875,0.6999999284744263,"('L', 'P', 'S')",0;1;2,...,537.2021484375,11,2.501393864661455;532.1045522523523;0.15742798...,16.84203191508792;100.53858857544493;6.6992745...,0.017998816859959212;0.0014087530035894122;0.0...,3.471377333213538,100.53858857544492,16.746920825432174,1388.271152986152,36.29629629629629
4,volume-101.nii.gz,"(512, 512, 683)","(512, 512, 683)",-1024.0,1604.0,0.705078125,0.705078125,0.699999988079071,"('L', 'P', 'S')",0;1;2,...,131.69473266601562,17,30.34652072918415;0.690769272595644;0.00034799...,38.69932836655393;10.967596461740541;0.8726858...,0.003044611220089117;0.026491883099033386;1.0;...,0.8726858999236206,50.28923314763987,13.41995609997164,1206.4169626892124,33.33333333333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,volume-43.nii.gz,"(512, 512, 155)","(512, 512, 155)",-1024.0,1023.0,1.0,1.0,1.0,"('R', 'A', 'S')",0;1;2,...,7.336999893188477,1,7.337,24.108672584252307,0.01103143981082684,24.108672584252307,24.108672584252307,24.108672584252307,1541.9628018327344,31.11111111111111
127,volume-44.nii.gz,"(512, 512, 119)","(512, 512, 119)",-1024.0,1023.0,1.0,1.0,1.0,"('R', 'A', 'S')",0;1;2,...,185.9900054931641,2,96.961;89.029,56.99885800990996;55.40015217393808,0.003688976617744595;0.003396908260297482,55.40015217393808,56.99885800990996,56.19950509192402,893.292882314723,20.74074074074074
128,volume-46.nii.gz,"(512, 512, 124)","(512, 512, 124)",-1024.0,1023.0,1.0,1.0,1.0,"('R', 'A', 'S')",0;1;2,...,41.41600036621094,48,0.053;1.26;0.108;0.484;0.891;0.434;0.097;0.356...,4.66042742317893;13.400591678688436;5.90847013...,0.2371013323463995;0.015993158732448734;0.1163...,3.675571780648109,27.50402471144558,9.325740658138628,1130.0515513722023,19.25925925925926
129,volume-47.nii.gz,"(512, 512, 225)","(512, 512, 225)",-1024.0,1024.0,1.0,1.0,1.0,"('R', 'A', 'S')",0;1,...,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,410.07101120445225,18.51851851851852


In [69]:
# We concluded this column's data is unreliable
old_updated = full_old.drop(columns=["lesion_sphericities"])

In [70]:
## update with the new values
old_updated = update_values('liver_hu_p01', full_new, old_updated)
old_updated = update_values('liver_hu_p99', full_new, old_updated)
old_updated = update_values('status', full_new, old_updated)
old_updated = update_values('obs', full_new, old_updated)

['volume-0.nii.gz', 'volume-1.nii.gz', 'volume-10.nii.gz', 'volume-100.nii.gz', 'volume-101.nii.gz', 'volume-102.nii.gz', 'volume-103.nii.gz', 'volume-104.nii.gz', 'volume-105.nii.gz', 'volume-106.nii.gz', 'volume-107.nii.gz', 'volume-108.nii.gz', 'volume-109.nii.gz', 'volume-11.nii.gz', 'volume-110.nii.gz', 'volume-111.nii.gz', 'volume-112.nii.gz', 'volume-113.nii.gz', 'volume-114.nii.gz', 'volume-115.nii.gz', 'volume-116.nii.gz', 'volume-117.nii.gz', 'volume-118.nii.gz', 'volume-119.nii.gz', 'volume-12.nii.gz', 'volume-120.nii.gz', 'volume-121.nii.gz', 'volume-122.nii.gz', 'volume-123.nii.gz', 'volume-124.nii.gz', 'volume-125.nii.gz', 'volume-126.nii.gz', 'volume-127.nii.gz', 'volume-128.nii.gz', 'volume-129.nii.gz', 'volume-13.nii.gz', 'volume-130.nii.gz', 'volume-14.nii.gz', 'volume-15.nii.gz', 'volume-16.nii.gz', 'volume-17.nii.gz', 'volume-18.nii.gz', 'volume-19.nii.gz', 'volume-2.nii.gz', 'volume-20.nii.gz', 'volume-21.nii.gz', 'volume-22.nii.gz', 'volume-23.nii.gz', 'volume-24.

In [72]:
old_updated.to_csv('../files/stats/lits/per_case_summary.csv', index=False)